# SPX 0DTE 信号与风险复盘：截至 2026-07-22

可执行伴随分析；固定 cutoff，不读取 2026-07-23 的未完成 session。

## TL;DR

- 近三日正式 300 秒确认共 7 次：周一 0、周二 2、周三 5；全部在 RTH 外。
- 严格 production 回放只有 3 个 unique trade-ready、1 个 fill，不能据此证明 edge。
- Jul 22 bad intent 的历史统计从 raw n=63 修正为 semantic n=41；统计变准不等于可以解除风险门控。
- 缺少 7/20–7/22 broker fills/fees，约 $12,000 实际亏损不能在本 notebook 对账。

## Context & Methods

读取固定 cutoff backtest、正式 level outcomes、trade intents 与 pricing outcomes。标的 300 秒回报按信号方向签名；RTH 使用 America/New_York 09:30–16:00。历史 play stats 在 Jul 22 首个 trade-ready 的准确 as-of 上计算：先复现 raw event-row 口径，再按生产 helper 的 `first_touch_at + contract_id + play` 全局去重。数值聚合使用 NumPy；胜率不确定性使用 Wilson 95% 区间。

In [1]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime, timedelta
from pathlib import Path
from statistics import NormalDist
from zoneinfo import ZoneInfo

import numpy as np

REPO_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "spx_spark").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the spx-spark repository root")
sys.path.insert(0, str(REPO_ROOT / "src"))

from spx_spark.strategy_contract import pricing_outcome_semantic_key

DATA_ROOT = Path(os.environ.get("SPX_SPARK_DATA_ROOT", "/srv/data/spx-spark/data"))
BACKTEST_ROOT = DATA_ROOT / "reports/odte_level_backtest/diagnostic=2026-07-23-final/cutoff=2026-07-22"
CUTOFF = datetime.fromisoformat("2026-07-23T00:00:00+00:00")
BAD_INTENT_AS_OF = datetime.fromisoformat("2026-07-22T09:22:05.290029+00:00")
ET = ZoneInfo("America/New_York")

backtest = json.loads((BACKTEST_ROOT / "artifact.json").read_text(encoding="utf-8"))
print(f"cutoff={backtest['window']['cutoff_at']} sessions={backtest['window']['trading_days']}")

cutoff=2026-07-23T00:00:00+00:00 sessions=7


## Data

源数据是本地 append-only JSONL feature stores 与最终 backtest artifact。Broker statement inventory 只覆盖到 2026-07-16；为避免泄露账户标识，本 notebook 不输出文件名。

In [2]:
def read_jsonl(path: Path):
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            payload = json.loads(line)
            if isinstance(payload, dict):
                yield payload

recent = []
for day in ("2026-07-20", "2026-07-21", "2026-07-22"):
    path = DATA_ROOT / f"features/level_decision_outcomes/date={day}/outcomes.jsonl"
    if not path.exists():
        continue
    for row in read_jsonl(path):
        if row.get("horizon_seconds") != 300:
            continue
        confirmed = datetime.fromisoformat(row["confirmed_at"])
        if confirmed >= CUTOFF:
            continue
        et = confirmed.astimezone(ET)
        directional = float(row["return_bps"]) * (1.0 if row["direction"] == "up" else -1.0)
        in_rth = et.weekday() < 5 and (et.hour, et.minute) >= (9, 30) and (et.hour, et.minute) < (16, 0)
        recent.append({
            "session_date": day,
            "weekday": et.strftime("%A"),
            "confirmed_et": et.strftime("%Y-%m-%d %H:%M:%S ET"),
            "direction": row["direction"],
            "play": row["thesis"],
            "level_kind": row["level_kind"],
            "directional_return_bps": directional,
            "direction_correct": directional > 0.0,
            "rth": in_rth,
        })

weekday_summary = []
for weekday in ("Monday", "Tuesday", "Wednesday"):
    values = np.array([r["directional_return_bps"] for r in recent if r["weekday"] == weekday], dtype=float)
    weekday_summary.append({
        "weekday": weekday,
        "n": int(values.size),
        "correct": int(np.count_nonzero(values > 0.0)),
        "mean_directional_bps": None if values.size == 0 else float(np.mean(values)),
    })

trade_ready = backtest["trade_ready_decisions"]
print(json.dumps({"recent_events": recent, "weekday_summary": weekday_summary, "trade_ready": trade_ready}, ensure_ascii=False, indent=2))

{
  "recent_events": [
    {
      "session_date": "2026-07-21",
      "weekday": "Tuesday",
      "confirmed_et": "2026-07-21 00:41:19 ET",
      "direction": "down",
      "play": "breakout",
      "level_kind": "flip_low",
      "directional_return_bps": 0.735697,
      "direction_correct": true,
      "rth": false
    },
    {
      "session_date": "2026-07-21",
      "weekday": "Tuesday",
      "confirmed_et": "2026-07-21 00:51:29 ET",
      "direction": "down",
      "play": "breakout",
      "level_kind": "flip_low",
      "directional_return_bps": 0.535103,
      "direction_correct": true,
      "rth": false
    },
    {
      "session_date": "2026-07-22",
      "weekday": "Wednesday",
      "confirmed_et": "2026-07-22 05:21:50 ET",
      "direction": "up",
      "play": "breakout",
      "level_kind": "flip_high",
      "directional_return_bps": -1.000787,
      "direction_correct": false,
      "rth": false
    },
    {
      "session_date": "2026-07-22",
      "weekday": "We

In [3]:
earliest = BAD_INTENT_AS_OF - timedelta(days=20)
eligible_pricing = []
for path in sorted((DATA_ROOT / "features/pricing_outcomes").glob("date=*/outcomes.jsonl")):
    for row in read_jsonl(path):
        completed_raw = row.get("completed_at")
        horizon = (row.get("horizons") or {}).get("300")
        if row.get("touched") is not True or not completed_raw or not isinstance(horizon, dict):
            continue
        value = horizon.get("return_fraction")
        completed = datetime.fromisoformat(completed_raw)
        if not isinstance(value, (int, float)) or not earliest <= completed <= BAD_INTENT_AS_OF:
            continue
        eligible_pricing.append((row, float(value)))

target = ("level_breakout_call", "flip_high")
raw_returns = np.array([
    value for row, value in eligible_pricing
    if (row.get("play"), row.get("level_kind")) == target
], dtype=float)

seen = set()
semantic_returns_list = []
for row, value in eligible_pricing:
    identity = pricing_outcome_semantic_key(row) or str(row.get("key") or "").strip()
    if identity and identity in seen:
        continue
    if identity:
        seen.add(identity)
    if (row.get("play"), row.get("level_kind")) == target:
        semantic_returns_list.append(value)
semantic_returns = np.array(semantic_returns_list, dtype=float)

def wilson_95(successes: int, sample_count: int):
    z = NormalDist().inv_cdf(0.975)
    p = successes / sample_count
    denominator = 1.0 + z * z / sample_count
    center = (p + z * z / (2.0 * sample_count)) / denominator
    half = z * np.sqrt(p * (1.0 - p) / sample_count + z * z / (4.0 * sample_count**2)) / denominator
    return float(center - half), float(center + half)

def summarize(values: np.ndarray):
    wins = int(np.count_nonzero(values > 0.0))
    low, high = wilson_95(wins, int(values.size))
    return {
        "n": int(values.size),
        "wins": wins,
        "winrate": wins / int(values.size),
        "avg_return": float(np.mean(values)),
        "median_return": float(np.median(values)),
        "wilson_95": [low, high],
    }

stats_compare = {"raw_event_rows": summarize(raw_returns), "semantic_first_touch": summarize(semantic_returns)}
print(json.dumps(stats_compare, indent=2))

{
  "raw_event_rows": {
    "n": 63,
    "wins": 19,
    "winrate": 0.30158730158730157,
    "avg_return": -0.019559264512446238,
    "median_return": -0.03418803418803407,
    "wilson_95": [
      0.20237691162188498,
      0.4236037231355958
    ]
  },
  "semantic_first_touch": {
    "n": 41,
    "wins": 18,
    "winrate": 0.43902439024390244,
    "avg_return": 0.002768070011058209,
    "median_return": -0.0042194092827004814,
    "wilson_95": [
      0.29890131736550374,
      0.5895947277789393
    ]
  }
}


## Results

下面的断言把报告中的关键数值锁定到原始 stores。若未来数据被改写、cutoff 漂移或语义去重逻辑变化，notebook 会失败而不是静默给出不同结论。

In [4]:
assert backtest["window"]["as_of"] == "2026-07-22"
assert backtest["window"]["trading_days"] == 7
assert backtest["strategy_readiness"]["sessions"]["contract_consistent_complete"] == 0
assert len(trade_ready) == 3
assert sum(row["execution_result"] == "filled" for row in trade_ready) == 1
assert len(recent) == 7
assert sum(row["rth"] for row in recent) == 0
assert [(row["weekday"], row["n"], row["correct"]) for row in weekday_summary] == [
    ("Monday", 0, 0), ("Tuesday", 2, 2), ("Wednesday", 5, 1)
]
assert np.isclose(weekday_summary[1]["mean_directional_bps"], 0.6354, atol=1e-6)
assert np.isclose(weekday_summary[2]["mean_directional_bps"], -3.6937328, atol=1e-6)
assert stats_compare["raw_event_rows"]["n"] == 63
assert stats_compare["semantic_first_touch"]["n"] == 41
assert np.isclose(stats_compare["semantic_first_touch"]["winrate"], 18 / 41)
assert np.isclose(stats_compare["semantic_first_touch"]["avg_return"], 0.002768070011058209)
print("VALIDATED: cutoff, recent outcomes, trade-ready replay, and semantic statistics all match the report.")

VALIDATED: cutoff, recent outcomes, trade-ready replay, and semantic statistics all match the report.


In [5]:
confirmed_baseline = backtest["profiles"]["baseline"]["confirmed"]["variants"]["naked"]
gth_baseline = backtest["profiles"]["baseline"]["gth_dip"]["variants"]["naked"]
production = backtest["production_strategy_total"]["result"]
result_snapshot = {
    "confirmed_proxy": {"fills": confirmed_baseline["n"], "winrate": confirmed_baseline["winrate"], "median_pnl_usd": confirmed_baseline["median_pnl_usd"], "total_pnl_usd": confirmed_baseline["total_pnl_usd"]},
    "gth_proxy": {"fills": gth_baseline["n"], "winrate": gth_baseline["winrate"], "total_pnl_usd": gth_baseline["total_pnl_usd"]},
    "production": {"fills": production["n"], "skipped": production["skipped"], "total_pnl_usd": production["total_pnl_usd"]},
    "modeled_costs": {"commission": False, "explicit_slippage": False, "queue": False, "partial_fills": False, "market_impact": False},
    "broker_loss_reconciled": False,
}
print(json.dumps(result_snapshot, indent=2))

{
  "confirmed_proxy": {
    "fills": 13,
    "winrate": 0.38461538461538464,
    "median_pnl_usd": -160.0,
    "total_pnl_usd": 310.0
  },
  "gth_proxy": {
    "fills": 3,
    "winrate": 0.0,
    "total_pnl_usd": -440.0
  },
  "production": {
    "fills": 1,
    "skipped": {
      "entry_limit_not_reached": 1,
      "target_before_entry": 1
    },
    "total_pnl_usd": 800.0
  },
  "modeled_costs": {
    "commission": false,
    "explicit_slippage": false,
    "queue": false,
    "partial_fills": false,
    "market_impact": false
  },
  "broker_loss_reconciled": false
}


## Takeaways

1. 不做星期参数：Monday n=0、Tuesday n=2、Wednesday n=5，任何 weekday gate 都是明显过拟合。
2. 风险边界先于收益优化：RTH-only 与 RR>=1.00 能直接阻止 Jul 22 类型的 bad intent。
3. 统计先去重：raw n=63 被重复经济机会污染；semantic n=41 才是正确输入，但置信区间仍不足以证明 edge。
4. GTH 保持 shadow：exact entry/exit forward cohort 仍是 0/20。
5. 完成真实亏损归因所需的唯一外部输入，是覆盖 2026-07-20–2026-07-22 的 broker Activity/Flex CSV。